In [ ]:
# basics 
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib import cm
# models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
# other tools
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, classification_report, confusion_matrix
# data sets
from sklearn.datasets import load_iris

### Zadanie 2

In [ ]:
df = load_iris()
print(df.keys())
df = pd.DataFrame(
    np.concatenate((df['data'], df['target'].reshape(-1, 1)), axis=1), 
    columns=df['feature_names'] + ['target'])
df

Checking assumptions (collinearity, etc)

Creating / training model

In [ ]:
# parameters 
seed = 1333
test_size = 0.3
features_3 = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)']
features_4 = features_3 + ['petal width (cm)']
# splitting 
X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(
    df[features_3], df['target'], 
    test_size=test_size, stratify=df['target'], random_state=seed)
X_train_4, X_test_4, y_train_4, y_test_4 = train_test_split(
    df[features_4], df['target'], 
    test_size=test_size, stratify=df['target'], random_state=seed
)
# creating model
model_3 = LogisticRegression()
model_3.fit(X_train_3, y_train_3)
model_4 = LogisticRegression()
model_4.fit(X_train_4, y_train_4)

In [ ]:
print(classification_report(y_test_3, model_3.predict(X_test_3)))
print(confusion_matrix(y_test_3, model_3.predict(X_test_3)))
print(classification_report(y_test_4, model_4.predict(X_test_4)))
print(confusion_matrix(y_test_4, model_4.predict(X_test_4)))

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# punkty treningowe, kolor = klasa
scatter = ax.scatter(
    X_train_3.iloc[:, 0],  # sepal length
    X_train_3.iloc[:, 1],  # sepal width
    X_train_3.iloc[:, 2],  # petal length
    c=y_train_3, cmap='Set1', edgecolor='k', s=50, alpha=0.8
)

ax.set_xlabel(features_3[0])
ax.set_ylabel(features_3[1])
ax.set_zlabel(features_3[2])
ax.set_title("3D Scatter Plot – True Classes")
legend = ax.legend(*scatter.legend_elements(), title="Class")
ax.add_artist(legend)
plt.tight_layout()
plt.show()


### zadanie 4

In [ ]:
N=1000


df1 = 20*pd.DataFrame(np.random.randn(N, 4), columns=['A', 'B','C','D'])
df1['B']=df1['A']+40*np.random.normal(0,1,N)
df1['C']=df1['A'] ** 2 + 100*np.random.normal(0,1,N)

df2 = [10,30,300,0]*pd.DataFrame(np.random.randn(N, 4), columns=['A', 'B','C','D'])
df2['C']=df2['C']+1000

Data=np.concatenate((df1, df2), axis=0)
Data[:,3]=1
Data[0:N,3]=0


import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


X = Data[:,:3]
y = Data[:,3]

In [ ]:
# splitting data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, 
    stratify=y, random_state=seed)
# created squared data
X_train_sqrd, X_test_sqrd = X_train.copy(), X_test.copy()
X_train_sqrd[:, 0] =  X_train_sqrd[:, 0]**2
X_test_sqrd[:, 0]= X_test_sqrd[:, 0] **2

# training models
model_knn = KNeighborsClassifier()
model_knn.fit(X_train, y_train)
knn_predict = model_knn.predict(X_test)

model_lg = LogisticRegression()
model_lg.fit(X_train, y_train)
lg_predict = model_lg.predict(X_test)

model_lg_sqrd = LogisticRegression()
model_lg_sqrd.fit(X_train_sqrd, y_train)
lg_sqrd_predict = model_lg_sqrd.predict(X_test_sqrd)


In [ ]:
print('KNN report\n')
print(classification_report(y_test, knn_predict))
print('Logistic regression report\n')
print(classification_report(y_test, lg_predict))
print('Logistics regression squared report\n')
print(classification_report(y_test, lg_sqrd_predict))

Strategy 

we will use fact that border line is when $h_\beta(X) = \frac12$ so when $\sum_i \beta_i X_i = 0$. So we will get value for third third coefficent based on first two and the third coefficent will be our Z value. In case of squared model me need to square $x_2$

In [ ]:
# taking coefficients: 
fig, ax = plt.subplots(subplot_kw={'projection': '3d'})

# creating grid
dens = 50
x_range = np.linspace(X[:, 0].min(), X[:, 0].max(), dens)
y_range = np.linspace(X[:, 1].min(), X[:, 1].max(), dens)
x1, x2 = np.meshgrid(x_range, y_range)

# finding border for classical model 
coef, intercept = model_lg.coef_[0], model_lg.intercept_
Z_lg = -(coef[0] * x1 + coef[1] * x2 + intercept) / coef[2]
ax.plot_surface(x1, x2, Z_lg, color='blue', alpha=0.5, label='Linear model')

# finding border for quadratic model
coef_sqrd, intercept_sqrd = model_lg_sqrd.coef_[0], model_lg_sqrd.intercept_
Z_lg_sqrd = -(coef_sqrd[0] * x1**2 + coef_sqrd[0] * x2 + intercept_sqrd) / coef_sqrd[2]
ax.plot_surface(x1, x2, Z_lg_sqrd, color='red', alpha=0.5, label='quadratic model')

ax.scatter(X_train[:, 0], X_train[:, 1], X_train[:, 2], cmap='coolwarm')

ax.set_title("Border line or logistic model")
ax.set_xlabel("Feature A")
ax.set_ylabel("Feature B")
ax.set_zlabel("Feature C")
plt.show()